# TP — Construire gratuitement la base de données de **DisclosureDelta**

## Objectif

À la fin de ce TP, tu auras une base locale contenant :

1. le mapping **ticker ↔ CIK** de la SEC ;
2. les métadonnées des rapports **10-K** et **10-Q** ;
3. les fichiers HTML bruts des publications ;
4. une version texte nettoyée des documents ;
5. une première extraction des sections **Risk Factors** et **MD&A** ;
6. les fondamentaux SEC au format XBRL ;
7. les cours quotidiens, volumes, dividendes et splits via `yfinance` ;
8. les facteurs quotidiens Fama–French 5 et Momentum ;
9. une table événementielle avec les rendements futurs à 1, 5 et 20 jours.

Le TP commence sur 10 sociétés afin de valider le pipeline. Une fois les contrôles réussis, tu pourras passer à 100–200 sociétés sans modifier l'architecture.

## Règles importantes

- Les téléchargements SEC doivent contenir un `User-Agent` identifiable avec ton nom et ton adresse e-mail.
- Ne dépasse jamais 10 requêtes par seconde sur les sites de la SEC. Le code ci-dessous est volontairement plus lent.
- Les prix obtenus via `yfinance` conviennent à un prototype académique ou personnel, pas à une base institutionnelle point-in-time.
- Conserve systématiquement les données brutes : ne remplace jamais un fichier brut par une version transformée.

## Étape 0 — Installation

Exécute cette cellule une seule fois dans un nouvel environnement Python ou dans Google Colab.

In [ ]:
%pip install -q pandas pyarrow requests beautifulsoup4 lxml yfinance tqdm tenacity python-dotenv

## Étape 1 — Configuration et arborescence

Remplace obligatoirement l'adresse e-mail ci-dessous. Elle sera envoyée dans le `User-Agent` des requêtes SEC.

Pour le premier test, conserve les 10 tickers proposés. La période commence un an avant la période de recherche afin de disposer d'un historique de prix suffisant.

In [1]:
from __future__ import annotations

import io
import json
import re
import time
import zipfile
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tenacity import retry, stop_after_attempt, wait_exponential
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

PROJECT_ROOT = Path.cwd() / "disclosure_delta_data"

DIRS = {
    "raw_sec_submissions": PROJECT_ROOT / "data/raw/sec/submissions",
    "raw_sec_filings": PROJECT_ROOT / "data/raw/sec/filings",
    "raw_sec_companyfacts": PROJECT_ROOT / "data/raw/sec/companyfacts",
    "raw_prices": PROJECT_ROOT / "data/raw/prices",
    "raw_factors": PROJECT_ROOT / "data/raw/factors",
    "interim_text": PROJECT_ROOT / "data/interim/filing_text",
    "interim_sections": PROJECT_ROOT / "data/interim/sections",
    "processed": PROJECT_ROOT / "data/processed",
    "logs": PROJECT_ROOT / "logs",
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

# À MODIFIER IMPÉRATIVEMENT.
SEC_USER_AGENT = "DisclosureDelta Research Ben_selma hadylbenselma@gmail.com"

START_DATE = "2015-01-01"
END_DATE = "2026-01-01"
PRICE_START_DATE = "2014-01-01"

TEST_TICKERS = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META",
    "GOOGL", "JPM", "XOM", "JNJ", "PG",
]

print(PROJECT_ROOT)

c:\Users\MSI\Desktop\FinanceProjects\NLP_AlphaGeneration\disclosure_delta_data


## Étape 2 — Créer une session HTTP robuste

La session réessaie les erreurs transitoires, identifie le programme auprès de la SEC, attend entre les appels et vérifie les réponses.

In [ ]:
SEC_MIN_INTERVAL_SECONDS = 0.15
_last_sec_request_time = 0.0


def build_session() -> requests.Session:
    session = requests.Session()
    retries = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.headers.update(
        {
            "User-Agent": SEC_USER_AGENT,
            "Accept-Encoding": "gzip, deflate",
        }
    )
    return session


SESSION = build_session()


def sec_get(url: str, *, expect_json: bool = False) -> Any:
    global _last_sec_request_time

    elapsed = time.time() - _last_sec_request_time
    if elapsed < SEC_MIN_INTERVAL_SECONDS:
        time.sleep(SEC_MIN_INTERVAL_SECONDS - elapsed)

    response = SESSION.get(url, timeout=60)
    _last_sec_request_time = time.time()
    response.raise_for_status()
    return response.json() if expect_json else response


print("Session prête.")

## Étape 3 — Télécharger le mapping ticker–CIK

Le **CIK** est l'identifiant stable attribué par la SEC. Il est plus fiable que le ticker, qui peut changer.

In [ ]:
COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
mapping_path = DIRS["processed"] / "sec_company_tickers.parquet"

raw_mapping = sec_get(COMPANY_TICKERS_URL, expect_json=True)

ticker_mapping = (
    pd.DataFrame.from_dict(raw_mapping, orient="index")
    .rename(columns={"cik_str": "cik", "title": "company_name"})
)

ticker_mapping["ticker"] = ticker_mapping["ticker"].str.upper()
ticker_mapping["cik"] = ticker_mapping["cik"].astype(int)
ticker_mapping["cik10"] = ticker_mapping["cik"].astype(str).str.zfill(10)
ticker_mapping = ticker_mapping[["ticker", "cik", "cik10", "company_name"]]
ticker_mapping.to_parquet(mapping_path, index=False)

universe = ticker_mapping[ticker_mapping["ticker"].isin(TEST_TICKERS)].copy()
missing = sorted(set(TEST_TICKERS) - set(universe["ticker"]))

print(universe.sort_values("ticker").to_string(index=False))
print("Tickers absents :", missing)
assert not missing, f"Tickers non résolus : {missing}"

## Étape 4 — Télécharger l'historique des soumissions SEC

L'endpoint principal contient les dépôts récents. Lorsqu'une société a davantage d'historique, la fonction récupère également les fichiers complémentaires.

In [ ]:
SUBMISSIONS_TEMPLATE = "https://data.sec.gov/submissions/CIK{cik10}.json"
SUBMISSIONS_EXTRA_TEMPLATE = "https://data.sec.gov/submissions/{filename}"


def columnar_dict_to_frame(data: dict[str, list[Any]]) -> pd.DataFrame:
    lengths = [len(v) for v in data.values() if isinstance(v, list)]
    if not lengths:
        return pd.DataFrame()
    n = min(lengths)
    return pd.DataFrame({k: v[:n] for k, v in data.items() if isinstance(v, list)})


def download_company_submissions(cik10: str) -> tuple[dict[str, Any], pd.DataFrame]:
    path = DIRS["raw_sec_submissions"] / f"CIK{cik10}.json"

    if path.exists():
        root = json.loads(path.read_text(encoding="utf-8"))
    else:
        root = sec_get(SUBMISSIONS_TEMPLATE.format(cik10=cik10), expect_json=True)
        path.write_text(json.dumps(root), encoding="utf-8")

    frames = []
    recent_df = columnar_dict_to_frame(root.get("filings", {}).get("recent", {}))
    if not recent_df.empty:
        frames.append(recent_df)

    for extra in root.get("filings", {}).get("files", []):
        filename = extra.get("name")
        if not filename:
            continue
        extra_path = DIRS["raw_sec_submissions"] / filename
        if extra_path.exists():
            extra_json = json.loads(extra_path.read_text(encoding="utf-8"))
        else:
            extra_json = sec_get(
                SUBMISSIONS_EXTRA_TEMPLATE.format(filename=filename),
                expect_json=True,
            )
            extra_path.write_text(json.dumps(extra_json), encoding="utf-8")

        extra_df = columnar_dict_to_frame(extra_json)
        if not extra_df.empty:
            frames.append(extra_df)

    filings = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return root, filings


all_filings = []

for row in tqdm(universe.itertuples(index=False), total=len(universe)):
    _, filings = download_company_submissions(row.cik10)
    if filings.empty:
        continue
    filings["cik"] = row.cik
    filings["cik10"] = row.cik10
    filings["ticker"] = row.ticker
    filings["company_name"] = row.company_name
    all_filings.append(filings)

filings_metadata = pd.concat(all_filings, ignore_index=True)
filings_metadata["filingDate"] = pd.to_datetime(
    filings_metadata["filingDate"], errors="coerce"
)

filings_metadata = filings_metadata[
    filings_metadata["form"].isin(["10-K", "10-Q"])
    & filings_metadata["filingDate"].between(START_DATE, END_DATE, inclusive="left")
].copy()

filings_metadata = filings_metadata.sort_values(
    ["ticker", "filingDate", "accessionNumber"]
).drop_duplicates(["cik", "accessionNumber"])

metadata_path = DIRS["processed"] / "filings_metadata.parquet"
filings_metadata.to_parquet(metadata_path, index=False)

print(f"{len(filings_metadata):,} filings 10-K/10-Q sélectionnés.")
display(
    filings_metadata[
        ["ticker", "form", "filingDate", "accessionNumber", "primaryDocument"]
    ].head(15)
)

### Contrôle 4A — Vérifier les métadonnées

On s'attend approximativement à cinq publications par société et par an : un 10-K et plusieurs 10-Q.

In [ ]:
filing_counts = (
    filings_metadata
    .groupby(["ticker", "form"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

display(filing_counts)

assert filings_metadata["accessionNumber"].notna().all()
assert filings_metadata["primaryDocument"].notna().all()
assert not filings_metadata.duplicated(["cik", "accessionNumber"]).any()

## Étape 5 — Télécharger les documents HTML bruts

Le téléchargement reprend automatiquement : un fichier déjà présent n'est pas redemandé.

In [ ]:
def filing_document_url(cik: int, accession_number: str, primary_document: str) -> str:
    accession_compact = accession_number.replace("-", "")
    return (
        "https://www.sec.gov/Archives/edgar/data/"
        f"{int(cik)}/{accession_compact}/{primary_document}"
    )


def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value)


def download_filing_html(row: pd.Series) -> Path | None:
    ticker_dir = DIRS["raw_sec_filings"] / safe_filename(row["ticker"])
    ticker_dir.mkdir(parents=True, exist_ok=True)

    suffix = Path(str(row["primaryDocument"])).suffix or ".html"
    filename = (
        f"{row['filingDate'].date()}_{row['form']}_"
        f"{row['accessionNumber'].replace('-', '')}{suffix}"
    )
    path = ticker_dir / filename

    if path.exists() and path.stat().st_size > 0:
        return path

    url = filing_document_url(
        int(row["cik"]),
        str(row["accessionNumber"]),
        str(row["primaryDocument"]),
    )

    try:
        response = sec_get(url)
        path.write_bytes(response.content)
        return path
    except requests.RequestException as exc:
        print(f"Échec : {row['ticker']} {row['accessionNumber']} — {exc}")
        return None


# Pour un premier essai rapide :
# filings_to_download = filings_metadata.head(20)
filings_to_download = filings_metadata.copy()

local_paths = []
for _, filing in tqdm(
    filings_to_download.iterrows(),
    total=len(filings_to_download),
    desc="Téléchargement des filings",
):
    local_paths.append(download_filing_html(filing))

filings_to_download["local_html_path"] = [
    str(path) if path is not None else None for path in local_paths
]
filings_to_download.to_parquet(metadata_path, index=False)

print(
    "Taux de téléchargement :",
    f"{filings_to_download['local_html_path'].notna().mean():.1%}",
)

## Étape 6 — Nettoyer les documents et extraire les sections

La fonction suivante est une baseline. L'extraction parfaite des sections EDGAR nécessite ensuite un audit manuel et des améliorations.

In [ ]:
def html_to_clean_text(path: str | Path) -> str:
    raw = Path(path).read_bytes()
    soup = BeautifulSoup(raw, "lxml")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


SECTION_PATTERNS = {
    "10-K": {
        "risk_factors": (
            [r"\bITEM\s+1A[.\s:–-]+RISK\s+FACTORS\b"],
            [
                r"\bITEM\s+1B[.\s:–-]+",
                r"\bITEM\s+1C[.\s:–-]+",
                r"\bITEM\s+2[.\s:–-]+",
            ],
        ),
        "mda": (
            [r"\bITEM\s+7[.\s:–-]+MANAGEMENT['’]S\s+DISCUSSION"],
            [r"\bITEM\s+7A[.\s:–-]+", r"\bITEM\s+8[.\s:–-]+"],
        ),
    },
    "10-Q": {
        "mda": (
            [r"\bITEM\s+2[.\s:–-]+MANAGEMENT['’]S\s+DISCUSSION"],
            [r"\bITEM\s+3[.\s:–-]+", r"\bITEM\s+4[.\s:–-]+"],
        ),
        "risk_factors": (
            [r"\bITEM\s+1A[.\s:–-]+RISK\s+FACTORS\b"],
            [r"\bITEM\s+2[.\s:–-]+", r"\bITEM\s+3[.\s:–-]+"],
        ),
    },
}


def find_all_starts(text: str, patterns: Iterable[str]) -> list[int]:
    starts = []
    for pattern in patterns:
        starts.extend(match.start() for match in re.finditer(pattern, text, flags=re.I))
    return sorted(set(starts))


def extract_section(
    text: str,
    start_patterns: Iterable[str],
    end_patterns: Iterable[str],
    min_chars: int = 500,
    max_chars: int = 500_000,
) -> str | None:
    starts = find_all_starts(text, start_patterns)
    if not starts:
        return None

    for start in reversed(starts):
        candidate_ends = []
        for end_pattern in end_patterns:
            match = re.search(end_pattern, text[start + 1:], flags=re.I)
            if match:
                candidate_ends.append(start + 1 + match.start())

        end = min(candidate_ends) if candidate_ends else min(len(text), start + max_chars)
        section = text[start:end].strip()

        if min_chars <= len(section) <= max_chars:
            return section

    return None


section_rows = []

for _, row in tqdm(
    filings_to_download.dropna(subset=["local_html_path"]).iterrows(),
    total=filings_to_download["local_html_path"].notna().sum(),
    desc="Nettoyage et extraction",
):
    text = html_to_clean_text(row["local_html_path"])

    text_path = (
        DIRS["interim_text"]
        / row["ticker"]
        / f"{row['accessionNumber'].replace('-', '')}.txt"
    )
    text_path.parent.mkdir(parents=True, exist_ok=True)
    text_path.write_text(text, encoding="utf-8")

    patterns = SECTION_PATTERNS.get(row["form"], {})
    for section_name, (starts, ends) in patterns.items():
        section = extract_section(text, starts, ends)
        section_path = None

        if section:
            section_path = (
                DIRS["interim_sections"]
                / section_name
                / row["ticker"]
                / f"{row['accessionNumber'].replace('-', '')}.txt"
            )
            section_path.parent.mkdir(parents=True, exist_ok=True)
            section_path.write_text(section, encoding="utf-8")

        section_rows.append(
            {
                "ticker": row["ticker"],
                "cik": row["cik"],
                "form": row["form"],
                "filing_date": row["filingDate"],
                "accession_number": row["accessionNumber"],
                "section": section_name,
                "section_found": section is not None,
                "n_chars": len(section) if section else 0,
                "text_path": str(text_path),
                "section_path": str(section_path) if section_path else None,
            }
        )

sections_index = pd.DataFrame(section_rows)
sections_index.to_parquet(
    DIRS["processed"] / "sections_index.parquet",
    index=False,
)

display(
    sections_index.groupby(["form", "section"])["section_found"]
    .agg(["count", "mean"])
)

### Contrôle 6A — Audit manuel obligatoire

Lis au moins 20 sections choisies aléatoirement. Une section est valide si elle commence au bon titre, ne correspond pas au sommaire, n'englobe pas le rapport entier et se termine près de l'item suivant.

In [ ]:
sample_sections = (
    sections_index[sections_index["section_found"]]
    .sample(min(20, int(sections_index["section_found"].sum())), random_state=42)
)

display(
    sample_sections[
        ["ticker", "form", "filing_date", "section", "n_chars", "section_path"]
    ]
)

if not sample_sections.empty:
    example_path = sample_sections.iloc[0]["section_path"]
    example_text = Path(example_path).read_text(encoding="utf-8")
    print(example_text[:4_000])

## Étape 7 — Télécharger les fondamentaux XBRL de la SEC

Nous stockons d'abord le JSON brut, puis toutes les observations US-GAAP dans une table longue. La colonne `filed` préservera le caractère point-in-time.

In [ ]:
COMPANYFACTS_TEMPLATE = (
    "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"
)


def download_companyfacts(cik10: str) -> dict[str, Any]:
    path = DIRS["raw_sec_companyfacts"] / f"CIK{cik10}.json"

    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    payload = sec_get(
        COMPANYFACTS_TEMPLATE.format(cik10=cik10),
        expect_json=True,
    )
    path.write_text(json.dumps(payload), encoding="utf-8")
    return payload


def flatten_companyfacts(payload: dict[str, Any]) -> pd.DataFrame:
    rows = []
    entity_name = payload.get("entityName")
    cik = payload.get("cik")

    for taxonomy, concepts in payload.get("facts", {}).items():
        for concept, concept_data in concepts.items():
            for unit, observations in concept_data.get("units", {}).items():
                for obs in observations:
                    rows.append(
                        {
                            "cik": cik,
                            "entity_name": entity_name,
                            "taxonomy": taxonomy,
                            "concept": concept,
                            "label": concept_data.get("label"),
                            "description": concept_data.get("description"),
                            "unit": unit,
                            **obs,
                        }
                    )

    return pd.DataFrame(rows)


fundamental_frames = []

for row in tqdm(universe.itertuples(index=False), total=len(universe)):
    facts = flatten_companyfacts(download_companyfacts(row.cik10))
    if facts.empty:
        continue
    facts["ticker"] = row.ticker
    fundamental_frames.append(facts)

companyfacts_long = pd.concat(fundamental_frames, ignore_index=True)
companyfacts_long["filed"] = pd.to_datetime(companyfacts_long["filed"], errors="coerce")
companyfacts_long["end"] = pd.to_datetime(companyfacts_long["end"], errors="coerce")

companyfacts_long = companyfacts_long[
    companyfacts_long["filed"].between(START_DATE, END_DATE, inclusive="left")
].copy()

companyfacts_long.to_parquet(
    DIRS["processed"] / "companyfacts_long.parquet",
    index=False,
)

print(f"{len(companyfacts_long):,} observations XBRL.")
display(
    companyfacts_long[
        ["ticker", "concept", "unit", "val", "end", "filed", "form", "accn"]
    ].head()
)

### Contrôle 7A — Concepts comptables utiles

Recherche les concepts réellement utilisés avant d'imposer un nom unique. Les concepts usuels comprennent le chiffre d'affaires, le résultat net, les actifs, les passifs, les capitaux propres et la trésorerie.

In [ ]:
USEFUL_CONCEPT_PATTERNS = (
    "Revenue|NetIncomeLoss|Assets$|Liabilities$|StockholdersEquity|"
    "CashAndCashEquivalents"
)

useful_concepts = (
    companyfacts_long[
        companyfacts_long["concept"].str.contains(
            USEFUL_CONCEPT_PATTERNS,
            case=False,
            regex=True,
            na=False,
        )
    ]
    .groupby(["concept", "unit"])
    .size()
    .sort_values(ascending=False)
    .head(40)
)

display(useful_concepts)

## Étape 8 — Télécharger les cours quotidiens gratuitement

Les fichiers sont enregistrés ticker par ticker afin de reprendre facilement après une erreur. Pour les rendements, utilise en priorité `Adj Close` après contrôle de cohérence.

In [ ]:
@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=2, min=2, max=30),
    reraise=True,
)
def download_one_ticker(ticker: str, start: str, end: str) -> pd.DataFrame:
    history = yf.Ticker(ticker).history(
        start=start,
        end=end,
        interval="1d",
        auto_adjust=False,
        actions=True,
        repair=True,
        raise_errors=True,
    )

    if history.empty:
        raise ValueError(f"Aucune donnée retournée pour {ticker}")

    history = history.reset_index()
    history.columns = [
        str(column).strip().lower().replace(" ", "_")
        for column in history.columns
    ]

    date_col = "date" if "date" in history.columns else history.columns[0]
    history = history.rename(columns={date_col: "date"})
    history["date"] = pd.to_datetime(history["date"], utc=True).dt.tz_convert(None)
    history["ticker"] = ticker
    return history


price_frames = []
price_errors = []

for ticker in tqdm(TEST_TICKERS, desc="Cours quotidiens"):
    path = DIRS["raw_prices"] / f"{ticker}.parquet"

    try:
        if path.exists():
            prices = pd.read_parquet(path)
        else:
            prices = download_one_ticker(
                ticker,
                start=PRICE_START_DATE,
                end=END_DATE,
            )
            prices.to_parquet(path, index=False)

        price_frames.append(prices)
        time.sleep(0.5)
    except Exception as exc:
        price_errors.append({"ticker": ticker, "error": repr(exc)})

prices_daily = pd.concat(price_frames, ignore_index=True)
prices_daily = prices_daily.sort_values(["ticker", "date"])
prices_daily.to_parquet(
    DIRS["processed"] / "prices_daily.parquet",
    index=False,
)

print(f"{len(prices_daily):,} lignes de prix.")
print("Erreurs :", price_errors)
display(prices_daily.head())

### Contrôle 8A — Qualité des prix

Vérifie la couverture temporelle, les doublons, les prix non positifs, les volumes manquants et les variations extrêmes.

In [ ]:
price_audit = (
    prices_daily.groupby("ticker")
    .agg(
        start=("date", "min"),
        end=("date", "max"),
        n_days=("date", "size"),
        missing_close=("close", lambda x: x.isna().mean()),
        missing_volume=("volume", lambda x: x.isna().mean()),
    )
)

display(price_audit)

assert not prices_daily.duplicated(["ticker", "date"]).any()
assert (prices_daily["close"].dropna() > 0).all()

## Étape 9 — Télécharger les facteurs Fama–French

Nous téléchargeons les cinq facteurs quotidiens et le facteur Momentum. Les données sont converties de pourcentages en rendements décimaux.

In [ ]:
FF5_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
    "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
)
MOM_URL = (
    "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
    "F-F_Momentum_Factor_daily_CSV.zip"
)


def download_binary(url: str, path: Path) -> Path:
    if not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
    return path


def read_french_daily_zip(path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(path) as archive:
        csv_names = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]
        if len(csv_names) != 1:
            raise ValueError(f"CSV inattendus dans {path.name}: {csv_names}")
        raw = archive.read(csv_names[0]).decode("utf-8", errors="replace")

    lines = raw.splitlines()
    header_index = next(
        i for i, line in enumerate(lines)
        if re.match(r"^\s*,", line)
    )

    data_lines = [lines[header_index]]
    for line in lines[header_index + 1:]:
        if re.match(r"^\s*\d{8},", line):
            data_lines.append(line)
        elif len(data_lines) > 1:
            break

    frame = pd.read_csv(io.StringIO("\n".join(data_lines)))
    first_col = frame.columns[0]
    frame = frame.rename(columns={first_col: "date"})
    frame["date"] = pd.to_datetime(
        frame["date"].astype(str).str.strip(),
        format="%Y%m%d",
    )

    for column in frame.columns.drop("date"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce") / 100.0

    return frame


ff5_zip = download_binary(FF5_URL, DIRS["raw_factors"] / "ff5_daily.zip")
mom_zip = download_binary(MOM_URL, DIRS["raw_factors"] / "momentum_daily.zip")

ff5 = read_french_daily_zip(ff5_zip)
momentum = read_french_daily_zip(mom_zip)

momentum_value_columns = [c for c in momentum.columns if c != "date"]
if len(momentum_value_columns) != 1:
    raise ValueError(f"Colonnes Momentum inattendues : {momentum.columns.tolist()}")

momentum = momentum.rename(columns={momentum_value_columns[0]: "Mom"})
factors_daily = ff5.merge(momentum, on="date", how="outer").sort_values("date")
factors_daily = factors_daily[
    factors_daily["date"].between(PRICE_START_DATE, END_DATE, inclusive="left")
]

factors_daily.to_parquet(
    DIRS["processed"] / "factors_daily.parquet",
    index=False,
)

display(factors_daily.tail())

## Étape 10 — Construire une table événementielle minimale

Convention conservatrice : chaque publication est associée au premier jour de cotation strictement postérieur à sa date de filing. Une version ultérieure utilisera l'heure exacte d'acceptation.

In [ ]:
def build_forward_returns(
    prices: pd.DataFrame,
    horizons: tuple[int, ...] = (1, 5, 20),
) -> pd.DataFrame:
    output_frames = []
    price_column = "adj_close" if "adj_close" in prices.columns else "close"

    for ticker, group in prices.groupby("ticker", sort=False):
        group = group.sort_values("date").copy()
        group["entry_price"] = group[price_column]

        for horizon in horizons:
            group[f"ret_fwd_{horizon}d"] = (
                group[price_column].shift(-horizon) / group[price_column] - 1.0
            )

        output_frames.append(group)

    return pd.concat(output_frames, ignore_index=True)


prices_with_targets = build_forward_returns(prices_daily)
event_rows = []

for ticker, filings_group in filings_metadata.groupby("ticker"):
    px = prices_with_targets[
        prices_with_targets["ticker"] == ticker
    ].sort_values("date")

    if px.empty:
        continue

    merged = pd.merge_asof(
        filings_group.sort_values("filingDate"),
        px[
            [
                "date", "entry_price",
                "ret_fwd_1d", "ret_fwd_5d", "ret_fwd_20d",
            ]
        ],
        left_on="filingDate",
        right_on="date",
        direction="forward",
        allow_exact_matches=False,
    )
    event_rows.append(merged)

events = pd.concat(event_rows, ignore_index=True)
events = events.rename(columns={"date": "entry_date"})
events.to_parquet(
    DIRS["processed"] / "filing_events_with_returns.parquet",
    index=False,
)

display(
    events[
        [
            "ticker", "form", "filingDate", "entry_date",
            "ret_fwd_1d", "ret_fwd_5d", "ret_fwd_20d",
        ]
    ].head(20)
)

## Étape 11 — Audit final

Le pipeline est validé seulement si les clés ne sont pas dupliquées, les dates d'entrée sont postérieures aux filings, les facteurs couvrent les prix et les fichiers bruts sont encore présents.

In [ ]:
audit = {
    "n_companies": int(universe["ticker"].nunique()),
    "n_filings": int(filings_metadata["accessionNumber"].nunique()),
    "n_html_downloaded": int(filings_to_download["local_html_path"].notna().sum()),
    "n_sections_found": int(sections_index["section_found"].sum()),
    "section_success_rate": float(sections_index["section_found"].mean()),
    "n_price_rows": int(len(prices_daily)),
    "n_factor_rows": int(len(factors_daily)),
    "n_events": int(len(events)),
    "events_with_20d_return": int(events["ret_fwd_20d"].notna().sum()),
}

print(json.dumps(audit, indent=2))

assert not filings_metadata.duplicated(["cik", "accessionNumber"]).any()

dated_events = events.dropna(subset=["entry_date"])
assert (dated_events["entry_date"] > dated_events["filingDate"]).all()

assert factors_daily["date"].min() <= prices_daily["date"].min()
assert (PROJECT_ROOT / "data/raw/sec").exists()
assert (PROJECT_ROOT / "data/raw/prices").exists()

## Étape 12 — Passer de 10 à 150 sociétés

Procède par paliers :

1. **10 sociétés** : valider les URLs, formats et dates ;
2. **30 sociétés** : corriger les différences de mise en page ;
3. **100–150 sociétés** : constituer la première base de recherche.

Critères conseillés : sociétés américaines liquides, diversification sectorielle, plusieurs années de 10-K/10-Q et exclusion initiale des cas les plus difficiles.

Ne présente pas une liste actuelle d'indice comme un univers historique point-in-time. Cette limite doit être écrite explicitement dans le rapport.

## Étape 13 — Fichiers produits

```text
disclosure_delta_data/
├── data/
│   ├── raw/
│   │   ├── sec/
│   │   │   ├── submissions/
│   │   │   ├── filings/
│   │   │   └── companyfacts/
│   │   ├── prices/
│   │   └── factors/
│   ├── interim/
│   │   ├── filing_text/
│   │   └── sections/
│   └── processed/
│       ├── sec_company_tickers.parquet
│       ├── filings_metadata.parquet
│       ├── sections_index.parquet
│       ├── companyfacts_long.parquet
│       ├── prices_daily.parquet
│       ├── factors_daily.parquet
│       └── filing_events_with_returns.parquet
└── logs/
```

La prochaine étape sera la construction des features NLP : dictionnaire financier, sentiment, embeddings, semantic delta, puis validation walk-forward.

## Sources et documentation

- SEC EDGAR APIs : https://www.sec.gov/search-filings/edgar-application-programming-interfaces
- SEC Developer Resources : https://www.sec.gov/about/developer-resources
- SEC Fair Access : https://www.sec.gov/filergroup/announcements-old/new-rate-control-limits
- Kenneth French Data Library : https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
- `yfinance` : https://github.com/ranaroussi/yfinance

Avant de publier le dépôt GitHub, ajoute `data/` au `.gitignore`. Le dépôt doit contenir les scripts et instructions, pas l'ensemble des données brutes.